[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/scenegraph/scenegraph.ipynb)

# A Scenegraph, a Robot Arm and a Camera, as One Map

This example poses a five-part robot arm and photographs it through a two-lens camera. In a graphics pipeline every vertex goes through the same chain of steps: stretch a unit box into the shape of a part, place it where the arm's joints put it, move it into the camera's frame, send a ray from it through the lenses onto the sensor, and scale the sensor to pixels. The steps compose into one transformation per part before any vertex is touched, so each vertex costs a single application.

Every step here is a map in projective geometric algebra. A part's stretch is a map on points, `Point <- Point`; a motor places it, `pose >> scale`; the camera joins each point with the pupil into a ray, a map from points to lines, `Point & pupil`; a thin lens is its formula on rays, `Line - (center & (Line ^ plane)) / focal_length`, a map from lines to lines; the sensor meets each ray in a point, `Line ^ sensor`. Calling one map on another composes them, so the whole pipeline collapses to one map from a part's unit box to pixels.

In the matrix notation of computer graphics the collapsed map reads as the product of the viewport, projection, view and model matrices, $V P C M_i$, one per part, and in ray-transfer optics a thin lens as the matrix $\begin{pmatrix} 1 & 0 \\ -1/f & 1 \end{pmatrix}$ on a ray's height and angle.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
import numpy as np

from numga import stack
from numga import NumpyContext
from numga.algebras import PGA3D as ga
from examples.animation import save_figure, show_animation
from examples.geometry.scenegraph import render, scenarios
from examples.geometry.scenegraph.core import canonical_unit_box

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Plane = ga.gatype.vector()
Line = ga.gatype.antibivector()
Point = ga.gatype.antivector()
Motor = ga.gatype.rotor()
# Extensors (linear maps between blade subspaces): collineations of points, and optical maps of rays.
PointMap = ga.gatype((Point, Point))        # Point <- Point
LineMap = ga.gatype((Line, Line))           # Line <- Line
origin = mv.w.dual()

## 1. Unit boxes, stretched and placed

Every part of the arm is one unit cube, stretched by a map on points that sends each coordinate to its scaled axis and keeps the weight, then placed by the motor of its joint. Each joint's pivot is the product of the motors before it: a lift and a yaw at the base, then a lift and a pitch at the shoulder, the elbow and the wrist. A motor applied to a map moves what the map returns: the part still takes points of the unit cube, but returns them stretched and placed.

In [ ]:
# The arm's pose: base yaw, shoulder, elbow and wrist pitch.
pose_angles = (0.35, -0.45, 0.85, -0.40)

In [ ]:
def anisotropic_scale(scale_x: float, scale_y: float, scale_z: float) -> PointMap:
    """Anisotropic scaling extensor (Point <- Point) along the principal axes.

    The coordinate planes read a point's coordinates, `mv.x & point` being its x; each
    coordinate is sent to its scaled axis direction, and the weight to the origin.
    """
    coordinate_planes = mv.vector(np.eye(4))                                  # [coordinates] Plane
    scaled_axes = mv.antivector(np.diag([scale_x, scale_y, scale_z, 1.0]))   # [coordinates] Point
    return (scaled_axes * (coordinate_planes & Point)).sum(axis=0)


def robot_arm(joint_angles: tuple) -> tuple[PointMap, list[Motor]]:
    """Forward kinematics of a 5-body articulated robot arm.

    Every body is the canonical unit box shaped by an anisotropic scaling extensor and
    placed into the world by its cumulative joint motor. The joint angles are base yaw,
    shoulder pitch, elbow pitch and wrist pitch; the result is the batch of five
    unit-box-to-world maps and the four joint pivots.
    """
    theta_base, theta_shoulder, theta_elbow, theta_wrist = joint_angles
    # A translator by d along z is (mv.zw * d / 2).exp(), a rotor by theta about z is (mv.xy * theta / 2).exp().

    # Body 0: Base pedestal (fixed stationary block resting on the floor z == 0)
    body_0 = (mv.zw * 0.125 / 2).exp() >> anisotropic_scale(0.9, 0.9, 0.25)

    # Turret base pivot at height 0.25, yawing around vertical z-axis:
    pivot_turret = (mv.zw * 0.25 / 2).exp() * (mv.xy * theta_base / 2).exp()

    # Body 1: Turret rotating body
    body_1 = (pivot_turret * (mv.zw * 0.175 / 2).exp()) >> anisotropic_scale(0.5, 0.5, 0.35)

    # Shoulder pivot 0.35 above turret base, pitching around local y-axis:
    pivot_shoulder = pivot_turret * (mv.zw * 0.35 / 2).exp() * (mv.zx * theta_shoulder / 2).exp()

    # Body 2: Upper arm (anisotropically elongated along arm length)
    body_2 = (pivot_shoulder * (mv.zw * 0.6 / 2).exp()) >> anisotropic_scale(0.24, 0.24, 1.2)

    # Elbow pivot at tip of upper arm (1.2 units along upper arm), pitching around local y-axis:
    pivot_elbow = pivot_shoulder * (mv.zw * 1.2 / 2).exp() * (mv.zx * theta_elbow / 2).exp()

    # Body 3: Forearm (anisotropically elongated along forearm length)
    body_3 = (pivot_elbow * (mv.zw * 0.5 / 2).exp()) >> anisotropic_scale(0.18, 0.18, 1.0)

    # Wrist pivot at tip of forearm (1.0 units along forearm), pitching around local y-axis:
    pivot_wrist = pivot_elbow * (mv.zw * 1.0 / 2).exp() * (mv.zx * theta_wrist / 2).exp()

    # Body 4: Gripper tool block / end-effector
    body_4 = (pivot_wrist * (mv.zw * 0.15 / 2).exp()) >> anisotropic_scale(0.28, 0.14, 0.3)

    # Stack all 5 body extensors into a single batched extensor:
    bodies_to_world = stack([body_0, body_1, body_2, body_3, body_4])
    joint_pivots = [pivot_turret, pivot_shoulder, pivot_elbow, pivot_wrist]
    return bodies_to_world, joint_pivots


unit_box = canonical_unit_box()                                                # [vertices] Point
bodies_to_world, pivots = robot_arm(pose_angles)                               # [bodies] Point <- Point
world_vertices = bodies_to_world[:, None](unit_box[None, :])                   # [bodies, vertices] Point

## 2. A two-lens camera

A thin lens shears each ray by its incidence with the lens's centre; the relay lens is the same map carried a little down the axis by a motor. The camera pulls world points into its own frame, joins each with a point of the pupil into a ray, refracts the ray through both lenses and meets it with the sensor. A viewport, a stretch and a shift, maps the sensor's physical size to its pixels, rows counted downward.

In [ ]:
# The camera's eye and target, the lenses' focal lengths and the relay's place, the pupil point's offset,
# the sensor's distance, and the sensor's size and pixels.
eye, target = np.array([0.0, -3.6, 1.3]), np.array([0.0, 0.0, 1.18])
front_focal, rear_focal, rear_gap = 1.0, 0.8, -0.3
pupil_offset, sensor_distance = 0.02, 1.25
sensor_size, pixels = (1.6, 1.2), (640, 480)

In [ ]:
def look_at(position: np.ndarray, target: np.ndarray) -> Motor:
    """The camera's motor in the world, its optical axis -z towards the target."""
    translation = ((mv(Plane.derive.nondegenerate, position) ^ mv.w) / 2).exp()
    # Pitch the optical axis from horizontal towards the target: a quarter turn about x turns -z forward, along +y,
    # and the pitch tilts it down.
    delta = target - position
    pitch = np.arctan2(delta[2], delta[1])
    return translation * (mv.yz * (np.pi / 2 - pitch) / 2).exp()


def thin_lens(center: Point, plane: Plane, focal_length: float) -> LineMap:
    """Thin lens extensor (Line <- Line) focusing rays towards its center.

    Following Gaussian optics in PGA, a thin lens shears a line proportionally to its
    incidence with the optical center: `Line - (center & (Line ^ plane)) / focal_length`.
    """
    return Line - (center & (Line ^ plane)) / focal_length


def lens_train(focal_front: float, focal_rear: float, rear_gap: float):
    """Front and rear lenses on the optical axis -z, and the rear lens plane.

    The front lens sits at the origin. The rear lens is placed along the optical axis by
    transforming the lens extensor with its displacement motor:
    `rear_lens = rear_placement >> thin_lens(origin, front_plane, focal_rear)(rear_placement << Line)`.
    """
    front_plane = -mv.z
    front_lens = thin_lens(origin, front_plane, focal_front)
    rear_placement = (mv.zw * rear_gap / 2).exp()
    rear_lens = rear_placement >> thin_lens(origin, front_plane, focal_rear)(rear_placement << Line)
    return front_lens, rear_lens, rear_placement >> front_plane


def lens_camera(camera_pose: Motor, front_lens: LineMap, rear_lens: LineMap, pupil: Point, sensor_plane: Plane) -> PointMap:
    """Compound multi-lens camera extensor mapping world points to sensor points in the camera frame.

    A compound optical system is the functional composition of the lens extensors,
    optics = rear_lens(front_lens).
    """
    # Transform world points into local camera space, then join with pupil to form rays:
    incoming_rays = (camera_pose << Point) & pupil     # Line <- Point
    to_sensor = Line ^ sensor_plane                    # Point <- Line
    return to_sensor(rear_lens(front_lens(incoming_rays)))


def viewport(width: int, height: int, sensor_width: float, sensor_height: float) -> PointMap:
    """Viewport extensor mapping metric sensor coordinates (x, y) to pixel coordinates (u, v).

    The physical sensor, sensor_width by sensor_height, is mapped to width by height pixels.
    Because raster pixel rows increase downward (row 0 at the top), the vertical axis is reflected.
    """
    scale = anisotropic_scale(width / sensor_width, -height / sensor_height, 1.0)
    return ((mv.xw * width / 2 + mv.yw * height / 2) / 2).exp() >> scale


camera_pose = look_at(eye, target)                                             # [] Motor
front_lens, rear_lens, rear_plane = lens_train(front_focal, rear_focal, rear_gap)   # [] Line <- Line each
pupil = (mv.xw * pupil_offset).exp() >> origin                                 # [] Point
sensor_plane = mv.z + sensor_distance * mv.w                                   # [] Plane
camera = lens_camera(camera_pose, front_lens, rear_lens, pupil, sensor_plane)  # [] Point <- Point
world_to_pixel = viewport(*pixels, *sensor_size)(camera)                       # [] Point <- Point

## 3. The whole pipeline as one map

Composing the world-to-pixel map with each part's placement gives one map per part, from the unit cube straight to pixels, and every vertex of every part goes through it in one pass. The same pixels come out of applying the steps one after another. Left, the arm in the world with the camera and three rays traced from gripper corners through both lenses to the sensor; right, the photograph.

In [ ]:
def project_vertices(local_to_pixel: PointMap, unit_box: Point) -> Point:
    """Batch-project canonical unit box vertices to screen pixels with unit weight."""
    raw_pixels = local_to_pixel[:, None](unit_box[None, :])
    # Perspective normalization:
    return raw_pixels / (mv.w & raw_pixels)


def trace_rays(
    scene_points: Point,
    camera_pose: Motor,
    front_lens: LineMap,
    rear_lens: LineMap,
    rear_plane: Plane,
    pupil: Point,
    sensor_plane: Plane,
):
    """The world points where rays from scene points through the pupil cross each optical interface.

    Returns, per scene point, the scene point, the pupil on the front lens, the hit on the
    rear lens plane, and the hit on the sensor.
    """
    ray_in = (camera_pose << scene_points) & pupil

    # Refraction through front lens:
    ray_mid = front_lens(ray_in)
    rear_hit = ray_mid ^ rear_plane

    # Refraction through rear lens:
    sensor_hit = rear_lens(ray_mid) ^ sensor_plane

    # Transform all points back to world space:
    return (
        scene_points,
        (camera_pose >> pupil).broadcast_to(scene_points.shape),
        camera_pose >> (rear_hit / (mv.w & rear_hit)),
        camera_pose >> (sensor_hit / (mv.w & sensor_hit)),
    )


local_to_pixel = world_to_pixel(bodies_to_world)                               # [bodies] Point <- Point
projected_pixels = project_vertices(local_to_pixel, unit_box)                  # [bodies, vertices] Point
rays = trace_rays(world_vertices[-1, [7, 6, 2]], camera_pose, front_lens, rear_lens, rear_plane, pupil, sensor_plane)
save_figure(render.draw_scenegraph(world_vertices, projected_pixels, camera_pose, rays), "scenegraph");

In [ ]:
sequential = world_to_pixel(world_vertices)
sequential = sequential / (mv.w & sequential)

## 4. The arm in motion

Over a loop of joint angles the arm swings and the camera photographs each pose through the same collapsed pipeline, a new map per part per frame, with the ray from the gripper's tip traced through the lenses.

In [ ]:
show_animation(render.animate_scenegraph(scenarios.robot_sweep(scenarios.N_FRAMES)), "scenegraph", scenarios.FRAME_DURATION)

In [ ]:
# checks
# The collapsed map agrees with applying kinematics, camera and viewport in turn.
assert (sequential - projected_pixels).dual().norm_squared().to_array().max() < 1e-16